# Valores iniciais com os quatro métodos finais

Este estudo usa somente AST original, Jacobiana, Shao-Wu com `tanh(u) * tanh(v)` nos produtos de lags, e Heaps. Cada método recebe o mesmo banco determinístico de mecanismos em todas as condições; o ruído é gerado em cada passo. Os três notebooks históricos permanecem intactos.

In [1]:
from dataclasses import asdict
from pathlib import Path
import hashlib
import importlib.metadata
import json
import platform
import subprocess
import sys
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
            if (p / "src" / "tsfmxai").exists())
sys.path.insert(0, str(ROOT / "src"))
from tsfmxai.evaluation.initialization import (
    GROUPS, InitializationStudyConfig, generator_configs, sample_study_mechanisms,
    run_initialization_cases, run_history_case, run_forgetting_cases, run_history_pair,
)
from tsfmxai.generators.initialization import HistorySpec

CONFIG = InitializationStudyConfig(mechanisms=32, trajectories=3,
                                   mechanism_seed=2026111001, trajectory_seed=2026111002,
                                   innovation_std=.05, example_mechanisms=0)
assert GROUPS == ("Original AST", "Jacobian", "Shao-Wu", "Heaps")
BANKS = {group: sample_study_mechanisms(group, CONFIG) for group in GROUPS}
assert all(len(bank) == CONFIG.mechanisms for bank in BANKS.values())
assert all(m.certificate.holds for group, bank in BANKS.items() if group != "Original AST"
           for m in bank)
print("Mechanism bank:", {group: len(bank) for group, bank in BANKS.items()})

STUDY = "starting_values"
results = {group: run_initialization_cases(group, BANKS[group], CONFIG,
                                          study="starting_values") for group in GROUPS}
cosine_specs = (
    HistorySpec("cosine full", "cosine", amplitude=.5, period=48, fraction=1.),
    HistorySpec("cosine half", "cosine", amplitude=.5, period=48, fraction=.5),
)
for spec in cosine_specs:
    for group in GROUPS:
        result = run_history_case(group, BANKS[group], CONFIG, spec)
        results[group]["rows"].extend(result["rows"])
        results[group]["skipped"].extend(result["skipped"])
rows = pd.DataFrame(row for result in results.values() for row in result["rows"])
skipped = [row for result in results.values() for row in result["skipped"]]
summary = rows.groupby(["method", "case", "horizon"], sort=False).agg(
    attempts=("exploded", "size"), explosions=("exploded", "sum"),
    near_zero=("near_zero", "sum"),
).reset_index()
summary["explosion_rate"] = summary.explosions / summary.attempts
summary["near_zero_rate"] = summary.near_zero / summary.attempts
assert set(summary.method) == set(GROUPS)
assert summary.attempts.eq(CONFIG.mechanisms * CONFIG.trajectories).all()
display(summary)
display(pd.DataFrame(skipped))


Mechanism bank: {'Original AST': 32, 'Jacobian': 32, 'Shao-Wu': 32, 'Heaps': 32}


,method,case,horizon,attempts,explosions,near_zero,explosion_rate,near_zero_rate
0,Original AST,zero,96,96,22,10,0.229167,0.104167
1,Original AST,zero,192,96,40,11,0.416667,0.114583
2,Original AST,zero,384,96,47,7,0.489583,0.072917
3,Original AST,positive constant,96,96,25,7,0.260417,0.072917
4,Original AST,positive constant,192,96,45,8,0.468750,0.083333
...,...,...,...,...,...,...,...,...
70,Heaps,cosine full,192,96,0,29,0.000000,0.302083
71,Heaps,cosine full,384,96,0,33,0.000000,0.343750
72,Heaps,cosine half,96,96,0,32,0.000000,0.333333
73,Heaps,cosine half,192,96,0,29,0.000000,0.302083


,method,case,reason
0,Original AST,stationary joint,Stationary initializer unavailable
1,Jacobian,stationary joint,Stationary initializer unavailable
2,Shao-Wu,stationary joint,Stationary initializer unavailable


## Registro completo do experimento

A célula seguinte registra pergunta, hipótese, configuração, procedência, sementes,
métricas, resumo científico e comando de reprodução. Todos os resultados são
observações em horizontes finitos. Uma trajetória inválida permanece no denominador;
casos incompatíveis com o certificado são registrados como não executados.


In [2]:
SOURCE_PATHS = [
    "src/tsfmxai/evaluation/initialization.py",
    "src/tsfmxai/generators/initialization.py",
    "src/tsfmxai/generators/random_expression.py",
    "src/tsfmxai/generators/contraction.py",
    "src/tsfmxai/generators/shao_wu.py",
    "src/tsfmxai/generators/heaps.py",
    "src/tsfmxai/generators/stochastic.py",
    "src/tsfmxai/evaluation/stability.py",
]
record = {
    "experiment_id": '01_starting_values',
    "status": "executed",
    "research_question": 'Como a forma do histórico inicial afeta explosão e baixa amplitude nos quatro métodos finais?',
    "hypothesis": 'A forma inicial altera os resultados de horizonte finito sem alterar as equações amostradas.',
    "configuration": {
        "study": STUDY,
        "config": asdict(CONFIG),
        "generator_configs": {group: asdict(cfg) for group, cfg in generator_configs(CONFIG).items()},
        "cases": ["zero", "positive constant", "negative constant", "uniform", "stationary joint", *[asdict(s) for s in cosine_specs]],
        "noise_policy": "one explicit innovation stream per mechanism and trajectory; uniform bounded for Jacobian, Gaussian for other methods",
        "denominator_policy": "all attempted trajectories or pairs; unsupported startup laws and certified-domain violations listed in skipped",
    },
    "seed_policy": {
        "mechanism_root_seed": CONFIG.mechanism_seed,
        "trajectory_root_seed": CONFIG.trajectory_seed,
        "mechanism": "derive_sample_seed(mechanism_seed, mechanism_index)",
        "initialization_noise_stationary": "SeedSequence([trajectory_seed, mechanism_index, trajectory_index]).spawn(3)",
        "mechanism_bank_reused_across_cases": True,
    },
    "provenance": {
        "git_head": subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=ROOT, text=True).strip(),
        "dirty_worktree_at_execution": bool(subprocess.check_output(["git", "status", "--porcelain"], cwd=ROOT)),
        "python": sys.version,
        "platform": platform.platform(),
        "packages": {p: importlib.metadata.version(p) for p in ("numpy", "pandas", "nbformat", "nbclient")},
        "source_sha256": {p: hashlib.sha256((ROOT/p).read_bytes()).hexdigest() for p in SOURCE_PATHS},
    },
    "metrics": {"summary": json.loads(summary.to_json(orient="records")),
                "skipped": skipped},
    "scientific_summary": "At horizon 384: " + "; ".join(f"{g}: {int(summary[summary.method.eq(g) & summary.horizon.eq(384)].explosions.sum())} failures" for g in GROUPS) + f". {len(skipped)} unsupported blocks are listed separately. Rates depend on the startup law and sampled mechanisms.",
    "limitations": "32 mechanisms per family and three paths per mechanism; finite horizons; different innovation laws reflect each method's construction; no claim for arbitrary coupled systems.",
    "reproduction": "From repository root: python -m jupyter nbconvert --to notebook --execute experiments/time_series_initialization/final_methods/01_starting_values.ipynb --output 01_starting_values.ipynb --output-dir experiments/time_series_initialization/final_methods --ExecutePreprocessor.timeout=1800",
}
print(json.dumps(record, ensure_ascii=False, indent=2, default=lambda v: float(v) if isinstance(v, np.floating) else int(v) if isinstance(v, np.integer) else list(v)))


{
  "experiment_id": "01_starting_values",
  "status": "executed",
  "research_question": "Como a forma do histórico inicial afeta explosão e baixa amplitude nos quatro métodos finais?",
  "hypothesis": "A forma inicial altera os resultados de horizonte finito sem alterar as equações amostradas.",
  "configuration": {
    "study": "starting_values",
    "config": {
      "mechanisms": 32,
      "trajectories": 3,
      "mechanism_seed": 2026111001,
      "trajectory_seed": 2026111002,
      "horizons": [
        96,
        192,
        384
      ],
      "innovation_std": 0.05,
      "explosion_limit": 1000000.0,
      "near_zero_radius": 0.15,
      "tail_steps": 48,
      "flat_tolerance": 0.15,
      "window_steps": 24,
      "constant_value": 0.5,
      "uniform_radius": 1.0,
      "magnitude_scales": [
        0.0,
        0.1,
        0.5,
        1.0,
        2.0,
        4.0
      ],
      "forgetting_tolerance": 1e-06,
      "forgetting_tail": 24,
      "minimum_lag": 1,
    